# Run OpenTRIM (`opentrim`) from a Python notebook

This cell runs the `opentrim` executable with after generating a custom json input file. It also prints:
- the exit code
- stdout (normal output)
- stderr (error output)

It assumes `opentrim` is available on your `PATH`. If it is not, set `exe`
to the full path (example in the comment below).

In [64]:
from __future__ import annotations
from pathlib import Path
import subprocess


import json
import re
from pathlib import Path
from typing import Any, Dict, Union
from copy import deepcopy

PathLike = Union[str, Path]

The code below generates the template json file.

In [65]:
exe = "opentrim"

cmd_template = [exe, "-t"]

template_run = subprocess.run(
    cmd_template,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    timeout=20,
)
json_template = json.loads(template_run.stdout)

def _to_python_literals(s: str) -> str:
    # Convert JSON literals -> Python literals
    s = re.sub(r"\btrue\b", "True", s)
    s = re.sub(r"\bfalse\b", "False", s)
    s = re.sub(r"\bnull\b", "None", s)
    return s


def _is_simple_scalar(x: Any) -> bool:
    return x is None or isinstance(x, (str, int, float, bool))


def _format_inline_list(lst: list[Any]) -> str | None:
    """
    Return a one-line JSON representation for lst if it is "simple" enough.
    Otherwise return None.
    """
    # Only inline lists of scalars (numbers/strings/bools/None)
    if not all(_is_simple_scalar(x) for x in lst):
        return None
    # Use json.dumps so strings get double-quotes and floats get JSON formatting
    return json.dumps(lst, separators=(", ", ": "))


def pretty_format(
    obj: Any,
    *,
    indent: int = 4,
    level: int = 0,
    max_inline_list_items: int = 6,
    max_inline_list_width: int = 60,
    sort_keys: bool = False,
) -> str:
    """
    Pretty format a JSON-like object into a JSON-style string, but:
      - Dicts are multi-line with indentation
      - Lists that are "simple" + short are kept on one line, e.g. [100, 1, 1]
      - Other lists are expanded multi-line
    """
    pad = " " * (indent * level)
    pad_in = " " * (indent * (level + 1))

    if isinstance(obj, dict):
        items = list(obj.items())
        if sort_keys:
            items.sort(key=lambda kv: kv[0])

        if not items:
            return "{}"

        parts = ["{"]
        for i, (k, v) in enumerate(items):
            key = json.dumps(k)  # ensures double quotes
            val = pretty_format(
                v,
                indent=indent,
                level=level + 1,
                max_inline_list_items=max_inline_list_items,
                max_inline_list_width=max_inline_list_width,
                sort_keys=sort_keys,
            )
            comma = "," if i < len(items) - 1 else ""
            parts.append(f"{pad_in}{key}: {val}{comma}")
        parts.append(f"{pad}}}")
        return "\n".join(parts)

    if isinstance(obj, list):
        # Try inline list formatting
        if len(obj) <= max_inline_list_items:
            inline = _format_inline_list(obj)
            if inline is not None and len(inline) <= max_inline_list_width:
                return inline

        if not obj:
            return "[]"

        parts = ["["]
        for i, item in enumerate(obj):
            val = pretty_format(
                item,
                indent=indent,
                level=level + 1,
                max_inline_list_items=max_inline_list_items,
                max_inline_list_width=max_inline_list_width,
                sort_keys=sort_keys,
            )
            comma = "," if i < len(obj) - 1 else ""
            parts.append(f"{pad_in}{val}{comma}")
        parts.append(f"{pad}]")
        return "\n".join(parts)

    # Scalars
    return json.dumps(obj)


def dict_to_parameters_text(
    data: Dict[str, Any],
    *,
    var_name: str = "PARAMETERS",
    add_type_hint: bool = True,
    indent: int = 4,
    sort_keys: bool = False,
    max_inline_list_items: int = 6,
    max_inline_list_width: int = 60,
) -> str:
    body = pretty_format(
        data,
        indent=indent,
        level=0,
        max_inline_list_items=max_inline_list_items,
        max_inline_list_width=max_inline_list_width,
        sort_keys=sort_keys,
    )
    body = _to_python_literals(body)

    prefix = f"{var_name}: Dict[str, Any] = " if add_type_hint else f"{var_name} = "
    return prefix + body


PARAMETERS: Dict[str, Any] = deepcopy(json_template)

# Customize the parameters in the code below:

In [66]:
# Example edits:
PARAMETERS["Transport"]["flight_path_type"] = "Variable"

PARAMETERS["IonBeam"]["ion"]["atomic_mass"] = 1.00784
PARAMETERS["IonBeam"]["energy_distribution"]["center"] = 1000000.0
PARAMETERS["IonBeam"]["spatial_distribution"]["center"] = [
                0.0,
                5000.0,
                5000.0]
PARAMETERS["Target"]["size"] = [
            10000.0,
            10000.0,
            10000.0
        ]
PARAMETERS["Target"]["cell_count"] = [
            100,
            1,
            1
        ]
PARAMETERS["Target"]["materials"][0]["id"] = "Fe"
PARAMETERS["Target"]["materials"][0]["composition"][0]["Es"] = 3.0
PARAMETERS["Target"]["regions"][0]["material_id"] = "Fe"
PARAMETERS["Target"]["regions"][0]["size"] = [
            10000.0,
            10000.0,
            10000.0
        ]
PARAMETERS["Output"]["title"] = "1 MeV H on Fe example"
PARAMETERS["Output"]["outfilename"] = "Example1"
PARAMETERS["Output"]["store_exit_events"] = True
PARAMETERS["Output"]["store_pka_events"] = True

PARAMETERS["Run"]["threads"] = 4
PARAMETERS["Run"]["max_no_ions"] = 20000

PARAMETERS["UserTally"] = [] # otherwise it crashes

Below are the functions that generate the JSON file:

In [67]:
def build_config(params: Dict[str, Any]) -> Dict[str, Any]:
    """
    Hook to transform/validate parameters before writing.
    For now, it returns params unchanged.
    """
    return params

def write_config(
    params: Dict[str, Any],
    output_path: str | Path = "config.json",
    *,
    pretty: bool = True,
    sort_keys: bool = False,
) -> Path:
    """
    Notebook-friendly writer. Call this after editing `params` (e.g., PARAMETERS).

    Returns the Path to the written file.
    """
    config = build_config(params)

    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with out_path.open("w", encoding="utf-8") as f:
        json.dump(
            config,
            f,
            indent=4 if pretty else None,
            sort_keys=sort_keys,
        )
        f.write("\n")

    return out_path


def config_as_json_string(
    params: Dict[str, Any],
    *,
    pretty: bool = True,
    sort_keys: bool = False,
) -> str:
    """
    Useful in notebooks for display/debugging without writing to disk.
    """
    config = build_config(params)
    return json.dumps( #use this instead alone
        config,
        indent=4 if pretty else None, #keep this
        sort_keys=sort_keys,
    )
    

out = write_config(PARAMETERS, "config.json", pretty=True)
print("Wrote:", out)

Wrote: config.json


If your PATH is set up correctly, this is enough:

In [68]:
cmd = [exe, "-f", "config.json", "-o", "result", "-s", "42"]

result = subprocess.run(
    cmd,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    timeout=20,
)

print("returncode:", result.returncode)
print("stdout:\n", result.stdout)
print("stderr:\n", result.stderr)

returncode: 0
stdout:
 Parsing JSON config from config.json
Starting simulation '1 MeV H on Fe example'...


║                                        ║  0%║ETC 00:00:00║
║█████████████▋                          ║ 34%║ETC 00:00:01║
║███████████████████████████▋            ║ 69%║ETC 00:00:01║
║████████████████████████████████████████║100%║ETC 00:00:00║
║████████████████████████████████████████║100%║ETC 00:00:00║

Completed 20000 ion histories.
Threads: 4
Cpu time (s):  2.32489,	Ions/cpu-s:  8602.57
Real time (s): 0.611563,	Ions/real-s: 32703.1
Storing results in result.h5 ... OK.

stderr:
 
